# 03 · Dislocations and contrast

A dislocation bends the lattice around its core, so nearby voxels diffract at slightly different angles — that misorientation field is what DFXM images. Displacement fields are the classical isotropic-elasticity expressions ([Hirth & Lothe 1982](../docs/references.md#hirth-lothe-1982)); the imaging physics is [Borgi 2024](../docs/references.md#borgi-2024) Figs. 3–5. Everything below uses the closed-form resolution backend, so this notebook needs no kernel file.

In [ ]:
%matplotlib inline
import subprocess
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import PowerNorm

HERE = Path.cwd()
assert HERE.name == "examples", "Run this notebook from the examples/ folder"
IMG, OUT = HERE / "img", HERE / "out_03"
for d in (IMG, OUT):
    d.mkdir(exist_ok=True)

## Dislocation character: the α sweep

The identification engine sweeps the dislocation line direction by an in-plane rotation α (the `rotation_deg` label in its HDF5 output); the forward default `t = n×b` is the pure edge configuration. Same Burgers vector, same (111) slip plane — only α changes, and the contrast morphology changes with it.

In [ ]:
ID_TOML = """
mode = "single"

[reciprocal]
backend = "analytic"
beamstop = false

[crystal]
slip_plane_normal = [1, 1, 1]
sweep_all_slip_planes = false
b_vector_indices = [0]
angle_start_deg = 0.0
angle_stop_deg = 90.0
angle_step_deg = 45.0
exclude_invisibility = false

[scan.phi]
value = 1.75e-4

[io]
include_perfect_crystal = false
write_strain_provenance = false
"""
cfg_file = OUT / "alpha_sweep.toml"
cfg_file.write_text(ID_TOML, encoding="utf-8")
subprocess.run(
    [
        sys.executable,
        "-c",
        "from dfxm_geo.pipeline import cli_main_identify; cli_main_identify()",
        "--config",
        str(cfg_file),
        "--output",
        str(OUT / "alpha"),
    ],
    check=True,
)

alphas = [0, 45, 90]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
with h5py.File(OUT / "alpha" / "dfxm_identify.h5", "r") as f:
    frames = [f[f"/{scan}/instrument/dfxm_sim_detector/data"][0] for scan in ("1.1", "2.1", "3.1")]
vmax = max(fr.max() for fr in frames)
for ax, fr, a in zip(axes, frames, alphas, strict=False):
    ax.imshow(fr, cmap="magma", norm=PowerNorm(0.45, vmin=0.0, vmax=vmax))
    ax.set_title(f"α = {a}°" + ("  (pure edge)" if a == 0 else ""))
    ax.axis("off")
fig.tight_layout()

## Weak-beam contrast

Offsetting the goniometer from the Bragg peak suppresses the bulk background and leaves the steep strain near the core — frames far out on the rocking curve show the sparse bright-core "weak beam" look the identification pipeline exploits.

In [ ]:
from dfxm_geo.pipeline import SimulationConfig, run_simulation

ROCK_TOML = """
[reciprocal]
backend = "analytic"
beamstop = false

[scan.phi]
range = 1.25e-4
steps = 5

[io]
include_perfect_crystal = false
write_strain_provenance = false

[postprocess]
enabled = false
"""
cfg_file = OUT / "rocking.toml"
cfg_file.write_text(ROCK_TOML, encoding="utf-8")
res = run_simulation(SimulationConfig.from_toml(cfg_file), OUT / "rocking")
with h5py.File(res["h5_path"], "r") as f:
    frames = f["/1.1/instrument/dfxm_sim_detector/data"][...]

vmax = frames.max()
phis = np.linspace(-1.25e-4, 1.25e-4, frames.shape[0])
fig, axes = plt.subplots(1, frames.shape[0], figsize=(15, 3.2))
for ax, im, p in zip(axes, frames, phis, strict=False):
    ax.imshow(im, cmap="magma", norm=PowerNorm(0.45, vmin=0.0, vmax=vmax))
    ax.set_title(f"φ = {p * 1e6:+.0f} µrad")
    ax.axis("off")
fig.tight_layout()

## COM ≈ −qi: the README figure, executable

For a wall of edge dislocations, the φ/χ centre-of-mass maps of a mosaicity scan reproduce (negated) the geometrical-optics qi fields at the sample plane — the self-consistency result of [Borgi 2024](../docs/references.md#borgi-2024). This reproduces the same computation as the repository README figure; the scan ranges are the article's (φ ±600 µrad, χ ±2 mrad — radians in the TOML).

In [ ]:
from dfxm_geo.pipeline import run_postprocess

WALL_TOML = """
[reciprocal]
backend = "analytic"
beamstop = false

[crystal]
mode = "wall"

[crystal.wall]
dis = 4
ndis = 151
sample_remount = "S1"

[scan.phi]
range = 6e-4
steps = 11

[scan.chi]
range = 2e-3
steps = 11

[io]
include_perfect_crystal = true
"""
cfg_file = OUT / "wall.toml"
cfg_file.write_text(WALL_TOML, encoding="utf-8")
cfg = SimulationConfig.from_toml(cfg_file)
run_dir = OUT / "wall"
run_simulation(cfg, run_dir)
res = run_postprocess(run_dir, cfg)
phi_list, chi_list, qi_field = res["phi_list"], res["chi_list"], res["qi_field"]

In [ ]:
v = 1e-4
zmid = qi_field.shape[3] // 2
panels = [
    (r"$-COM_\varphi$", (-phi_list).T),
    (r"$-COM_\chi$", (-chi_list).T),
    (r"$q_{i,1}$", qi_field[0, :, :, zmid].squeeze()),
    (r"$q_{i,2}$", qi_field[1, :, :, zmid].squeeze()),
]
fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for ax, (title, data) in zip(axes.ravel(), panels, strict=False):
    pc = ax.imshow(data, vmin=-v, vmax=v, cmap="RdBu_r", origin="lower")
    ax.set_title(title)
    ax.axis("off")
fig.colorbar(pc, ax=axes, shrink=0.8, label="rad")
fig.savefig(IMG / "03_dislocations_preview.png", dpi=110, bbox_inches="tight")

Each top panel reproduces the one beneath it — the measured mosaicity maps recover the model's own input distortion field. Next: [04 · Oblique & reflections](04_oblique_and_reflections.ipynb).